# RAG Failure Localization and Evaluation

**Guiding question:** When a fluent answer is wrong, did retrieval supply the wrong evidence,
or did generation misuse the right evidence?

This notebook owns only RAG-stage diagnosis: retriever-versus-generator failure,
groundedness versus correctness, oracle-context ablation, metric fingerprints, per-query
records, citations, refusal, authorization, and safety gates. General judge bias,
hallucination taxonomies, and confidence calibration live in
[04-llm-evaluation](../04-llm-evaluation/README.md).


## Route

| Part | Falsifiable question | Evidence |
| --- | --- | --- |
| 1 | Can answer text identify the failing stage? | Four-cell failure matrix |
| 2 | Which component signal moved? | Per-query metric fingerprint |
| 3 | Would the same generator succeed with gold context? | Oracle-context ablation |
| 4 | Can a healthy mean hide a blocked release? | Independent gate record |

![Retriever-versus-generator failure map](images/rag-failure-location-map.png)


In [ ]:
# Deterministic setup and shared RAG evaluation fixtures.
import re
from dataclasses import asdict, dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rag_shared import COURSE_CAVEAT, RAG_EVAL_CASES, set_seed, tokenize

set_seed()
STOPWORDS = {"a", "an", "and", "are", "as", "be", "before", "does", "how", "is", "it", "or", "such", "the", "to"}
print(COURSE_CAVEAT)
print(f"Evaluation cases: {len(RAG_EVAL_CASES)}")


## Part 1 - One answer, two possible owners

A generator can faithfully summarize the wrong retrieved page. That answer is grounded to
its context and still incorrect. Conversely, a generator can receive the right page and add
an unsupported claim.

**Predict:** Which rows below should repair when retrieval is bypassed?


In [ ]:
# Four controlled states on one running ReAct question.
case = RAG_EVAL_CASES[0]
wrong_context = "LoRA adapts a model by training low-rank matrices while base weights stay frozen."
failure_cases = [
    {"name": "success", "retriever_ok": True, "generator_ok": True,
     "context": case["gold_context"], "answer": case["reference"]},
    {"name": "retriever_failure", "retriever_ok": False, "generator_ok": True,
     "context": wrong_context, "answer": "LoRA trains low-rank matrices while base weights remain frozen."},
    {"name": "generator_failure", "retriever_ok": True, "generator_ok": False,
     "context": case["gold_context"], "answer": "ReAct uses quantum entanglement to choose actions."},
    {"name": "double_failure", "retriever_ok": False, "generator_ok": False,
     "context": wrong_context, "answer": "ReAct stores every thought in a vector database."},
]
display(pd.DataFrame(failure_cases)[["name", "retriever_ok", "generator_ok", "answer"]])


In [ ]:
# Component metrics deliberately answer different questions.
def content_terms(text):
    return [token for token in tokenize(text) if token not in STOPWORDS]

def overlap_precision(answer, context):
    answer_terms = content_terms(answer)
    return sum(term in set(content_terms(context)) for term in answer_terms) / max(len(answer_terms), 1)

def overlap_f1(left, right):
    left_terms, right_terms = set(content_terms(left)), set(content_terms(right))
    overlap = len(left_terms & right_terms)
    precision = overlap / max(len(left_terms), 1)
    recall = overlap / max(len(right_terms), 1)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def retrieval_hit(context, gold_context):
    return float(context == gold_context)

def answer_relevance(question, answer):
    return overlap_f1(question, answer)

def correctness(answer, reference):
    return overlap_f1(answer, reference)


In [ ]:
# Read the metric fingerprint instead of averaging away the owner.
fingerprint_rows = []
for item in failure_cases:
    fingerprint_rows.append({
        "case": item["name"],
        "retrieval_hit": retrieval_hit(item["context"], case["gold_context"]),
        "groundedness": overlap_precision(item["answer"], item["context"]),
        "answer_relevance": answer_relevance(case["question"], item["answer"]),
        "correctness": correctness(item["answer"], case["reference"]),
    })
fingerprints = pd.DataFrame(fingerprint_rows).set_index("case")
display(fingerprints.round(3))

ax = fingerprints.plot.bar(figsize=(10, 4))
ax.set_ylim(0, 1.05)
ax.set_title("Failure fingerprints: same answer quality problem, different owner")
ax.set_ylabel("component score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Complaint:** groundedness alone rewards the faithful LoRA answer. Correctness alone cannot
explain why it failed. The fingerprint localizes the first broken stage.

These overlap metrics are cheap regression proxies. They miss negation, paraphrase, and
relation changes. The evaluation track owns richer judge and claim-level instruments.


## Part 2 - Preserve one record per query

Aggregates are useful only after every query retains its inputs, versions, component scores,
gate outcomes, and diagnosis.


In [ ]:
# One explicit schema prevents a dashboard from dropping the evidence needed for diagnosis.
@dataclass(frozen=True)
class RAGRecord:
    query_id: str
    retrieval_hit: float
    groundedness: float
    answer_relevance: float
    correctness: float
    citation_ok: bool
    refusal_ok: bool
    authorization_ok: bool
    safety_ok: bool
    diagnosis: str

def diagnose(scores):
    if scores["retrieval_hit"] == 0:
        return "retriever"
    if scores["groundedness"] < 0.5 or scores["correctness"] < 0.5:
        return "generator"
    return "healthy"


In [ ]:
# Build deterministic live-path records with one injected retrieval miss.
records = []
for index, item in enumerate(RAG_EVAL_CASES):
    context = wrong_context if index == 1 else item["gold_context"]
    answer = (
        "LoRA trains low-rank matrices while base weights stay frozen."
        if index == 1 else item["reference"]
    )
    scores = {
        "retrieval_hit": retrieval_hit(context, item["gold_context"]),
        "groundedness": overlap_precision(answer, context),
        "answer_relevance": answer_relevance(item["question"], answer),
        "correctness": correctness(answer, item["reference"]),
    }
    records.append(RAGRecord(
        query_id=item["id"], **scores,
        citation_ok=context == item["gold_context"],
        refusal_ok=True, authorization_ok=True, safety_ok=True,
        diagnosis=diagnose(scores),
    ))
record_table = pd.DataFrame([asdict(record) for record in records])
display(record_table)
assert record_table["query_id"].is_unique


## Part 3 - Oracle context isolates the bottleneck

Give the same answer constructor the labeled gold page. If answer scores jump, retrieval was
the limiting stage. If they do not, inspect generation or the evaluator.

![Oracle-context bottleneck isolation](images/oracle-context-bottleneck-isolation.gif)


In [ ]:
# A fixed constructor makes the ablation change only context.
def answer_from_context(item, context, stubborn=False):
    if stubborn:
        return "The answer is quantum entanglement."
    if context == item["gold_context"]:
        return item["reference"]
    return context.split(".")[0] + "."

ablation_rows = []
for index, item in enumerate(RAG_EVAL_CASES):
    live_context = wrong_context if index == 1 else item["gold_context"]
    stubborn = index == 2
    for path, context in [("live", live_context), ("oracle", item["gold_context"])]:
        answer = answer_from_context(item, context, stubborn=stubborn)
        ablation_rows.append({
            "query": item["id"], "path": path,
            "correctness": correctness(answer, item["reference"]),
            "groundedness": overlap_precision(answer, context),
        })
ablation = pd.DataFrame(ablation_rows)
display(ablation.pivot(index="query", columns="path", values="correctness").round(3))


In [ ]:
# The delta is the diagnostic, not a production quality score.
oracle_pivot = ablation.pivot(index="query", columns="path", values="correctness")
oracle_pivot["oracle_gain"] = oracle_pivot["oracle"] - oracle_pivot["live"]
oracle_pivot["owner"] = np.where(
    oracle_pivot["oracle_gain"] > 0.25, "retriever",
    np.where(oracle_pivot["oracle"] < 0.5, "generator/evaluator", "no isolated failure"),
)
display(oracle_pivot.round(3))
assert oracle_pivot.loc["rights-review", "owner"] == "retriever"
assert oracle_pivot.loc["parental-duration", "owner"] == "generator/evaluator"


## Part 4 - Independent release gates

Citation validity, refusal behavior, authorization, and safety are not bonus points in a
composite. One failed gate blocks release even when means look healthy.


In [ ]:
# Gate records remain booleans with explicit reasons.
gate_rows = [
    {"query": "react-tools", "citation": True, "refusal": True, "authorization": True, "safety": True},
    {"query": "rights-review", "citation": False, "refusal": True, "authorization": True, "safety": True},
    {"query": "unsupported-policy", "citation": True, "refusal": False, "authorization": True, "safety": True},
    {"query": "forbidden-policy", "citation": True, "refusal": True, "authorization": False, "safety": True},
]
gates = pd.DataFrame(gate_rows)
gate_columns = ["citation", "refusal", "authorization", "safety"]
gates["release"] = gates[gate_columns].all(axis=1)
gates["failed_gates"] = gates.apply(
    lambda row: [column for column in gate_columns if not row[column]], axis=1
)
display(gates)
assert not gates.query("query == 'forbidden-policy'").iloc[0]["release"]


In [ ]:
# Aggregate reporting comes last and keeps the failure count visible.
metric_columns = ["retrieval_hit", "groundedness", "answer_relevance", "correctness"]
summary = record_table[metric_columns].mean().to_frame("mean")
summary["queries"] = len(record_table)
summary["diagnosed_failures"] = (record_table["diagnosis"] != "healthy").sum()
display(summary.round(3))
print("Release gate pass rate:", gates["release"].mean())
print("A high metric mean cannot override a failed authorization, refusal, citation, or safety gate.")


## Scorecard and ownership boundary

- **Retriever failure:** low retrieval hit; oracle context creates a large correctness gain.
- **Generator failure:** retrieval is healthy; groundedness/correctness remain weak under oracle.
- **Metric fingerprint:** component shape localizes the repair; a composite hides it.
- **Per-query record:** versions, scores, gates, and diagnosis survive aggregation.
- **Release gates:** citation, refusal, authorization, and safety remain independent.

**Built and measured:** failure matrix, four-signal fingerprints, per-query records,
oracle-context gains, and gate decisions.

**Named and forwarded:** judge order/verbosity bias belongs to
[LLM-as-judge](../04-llm-evaluation/02-llm-as-judge-safety-and-pipeline.ipynb);
unsupported-claim localization belongs to
[hallucination detection](../04-llm-evaluation/03-hallucination-detection.ipynb);
confidence thresholds belong to
[calibration](../04-llm-evaluation/04-calibration-and-confidence.ipynb).
